# One-epoch GPU profile for Python/PyTorch full-basis NN training

This notebook profiles one non-Monte-Carlo GPU training epoch for the Python/PyTorch neural-network ansatz in `tJ/RC/tJNetRC.py`. It mirrors the GPU full-basis sweep settings and breaks the work into:

- CPU model/basis setup and Hamiltonian assembly
- PyTorch input construction and GPU transfer
- sparse Hamiltonian conversion with `tJNetRC.processH`
- network and optimizer setup
- NN forward pass
- sparse variational loss for a fixed `psi`
- backward pass through forward + variational loss
- optimizer update
- exact training-style `optimizer.step(closure)` timing

The default configuration is the 18-site, 2-hole sector with 8 spin-up and 8 spin-down electrons.

In [14]:
from __future__ import annotations

import gc
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List

import numpy as np
from scipy import sparse
import torch
from torch import nn


def find_tj_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "tJ", Path.cwd().parent / "tJ"]
    for directory in candidates:
        if (directory / "tJ.py").is_file() and (directory / "RC" / "tJNetRC.py").is_file():
            return directory.resolve()
    raise RuntimeError("Could not find tJ.py and RC/tJNetRC.py. Start from the project root or tJ/.")


TJ_DIR = find_tj_dir()
sys.path.insert(0, str(TJ_DIR))
sys.path.insert(0, str(TJ_DIR / "RC"))

from tJ import tJ  # noqa: E402
import tJNetRC as TJNetRC  # noqa: E402

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected by torch.cuda.is_available(). Run this on gpu_test.")

TORCH_DEVICE = torch.device("cuda")
TJNetRC.device = TORCH_DEVICE
torch.backends.cudnn.benchmark = False

RESULTS_DIR = TJ_DIR / "profile_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "gpu_one_epoch_profile_python.tsv"

{
    "TJ_DIR": str(TJ_DIR),
    "RESULTS_PATH": str(RESULTS_PATH),
    "torch_version": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(TORCH_DEVICE),
    "tJNetRC_device": str(TJNetRC.device),
}

{'TJ_DIR': '/n/holylabs/kaxiras_lab/axzhu/projects/tJ',
 'RESULTS_PATH': '/n/holylabs/kaxiras_lab/axzhu/projects/tJ/profile_results/gpu_one_epoch_profile_python.tsv',
 'torch_version': '2.11.0+cu129',
 'cuda_device': 'NVIDIA A100-SXM4-40GB MIG 3g.20gb',
 'tJNetRC_device': 'cuda'}

In [22]:
SITES = 16
HOLES = 2
NUP = 7
NDOWN = 7
LATTICE_SPEC = dict(m=4, n=0, m_=0, n_=2)

T_VALUE = 1.0
J_OVER_T = 0.3
TCOEF = -T_VALUE
JCOEF = J_OVER_T * T_VALUE

N_NEURONS = 128
DROPOUT = 0.0
LEARNING_RATE = 1.0e-3
OPTIMIZER_NAME = "Adam"
RNG_SEED = 12345
DTYPE_NP = np.float32
DTYPE_TORCH = torch.float32

# The Python sweep uses reordered basis. For profiling per-epoch GPU work, this is not necessary.
# Set True to match the sweep's basis order exactly, at the cost of a long setup step.
USE_REORDERED_BASIS = False

# Nearest-neighbor-only assembly matches the Ht/HJ passed to full-basis tJ training and skips unused Ht2/HJ2.
USE_NN_ONLY_HAMILTONIAN = True

# Fine-grained default: include forward-only and fixed-psi variational timings.
RUN_WARMUP_EPOCH = True
PROFILE_DIAGNOSTIC_SUBPIECES = True
PROFILE_DIAGNOSTIC_TRACK_GRAD = True
PROFILE_ACTUAL_CLOSURE_STEP = False
DROP_SCIPY_HAMILTONIAN_AFTER_GPU_TRANSFER = True

# trainNN_transfer currently uses retain_graph=True. Set this True for exact parity if memory allows.
PROFILE_RETAIN_GRAPH = False

# trainNN_transfer wraps the model in nn.DataParallel even on one visible GPU.
USE_DATA_PARALLEL = True

PARAMS = {
    "n_neurons": N_NEURONS,
    "dropout": DROPOUT,
    "optimizer": OPTIMIZER_NAME,
    "learning_rate": LEARNING_RATE,
    "adam_betas": (0.9, 0.999),
    "adam_eps": 1.0e-8,
    "adam_weight_decay": 0.0,
    "adam_amsgrad": False,
    "lbfgs_history_size": 200,
    "lbfgs_line_search_fn": "strong_wolfe",
    "exploit_symmetry": False,
}

{
    "sites": SITES,
    "holes": HOLES,
    "nup": NUP,
    "ndown": NDOWN,
    "J_over_t": J_OVER_T,
    "n_neurons": N_NEURONS,
    "optimizer": OPTIMIZER_NAME,
    "use_reordered_basis": USE_REORDERED_BASIS,
    "use_nn_only_hamiltonian": USE_NN_ONLY_HAMILTONIAN,
    "use_data_parallel": USE_DATA_PARALLEL,
    "profile_diagnostic_subpieces": PROFILE_DIAGNOSTIC_SUBPIECES,
    "profile_diagnostic_track_grad": PROFILE_DIAGNOSTIC_TRACK_GRAD,
    "profile_actual_closure_step": PROFILE_ACTUAL_CLOSURE_STEP,
    "profile_retain_graph": PROFILE_RETAIN_GRAPH,
}

{'sites': 16,
 'holes': 2,
 'nup': 7,
 'ndown': 7,
 'J_over_t': 0.3,
 'n_neurons': 128,
 'optimizer': 'Adam',
 'use_reordered_basis': False,
 'use_nn_only_hamiltonian': True,
 'use_data_parallel': True,
 'profile_diagnostic_subpieces': True,
 'profile_diagnostic_track_grad': True,
 'profile_actual_closure_step': False,
 'profile_retain_graph': False}

In [19]:
def timed_cuda(fn: Callable[[], Any]):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    result = fn()
    torch.cuda.synchronize()
    return result, time.perf_counter() - t0


def scalar_float(value: Any) -> float:
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu().item())
    return float(np.asarray(value).reshape(-1)[0])


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


PROFILE_DIM = 0
PROFILE_NNZ = 0
PROFILE_H_TYPE = ""
PROFILE_INPUT_TYPE = ""


def tsv_cell(value: Any) -> str:
    if value is None:
        return ""
    return str(value).replace("\t", " ").replace("\n", " ")


PROFILE_COLUMNS = [
    "timestamp", "stage", "seconds", "sites", "holes", "nup", "ndown", "dim", "nnz",
    "J_over_t", "n_neurons", "optimizer", "loss", "cuda_device", "H_type", "input_type", "notes",
]


def append_profile_rows(path: Path, rows: List[Dict[str, Any]]) -> Path:
    if not rows:
        return path
    new_file = (not path.exists()) or path.stat().st_size == 0
    with path.open("a") as handle:
        if new_file:
            handle.write("\t".join(PROFILE_COLUMNS) + "\n")
        for row in rows:
            handle.write("\t".join(tsv_cell(row.get(col, "")) for col in PROFILE_COLUMNS) + "\n")
    return path


def base_profile_row(stage: str, seconds: float, *, loss: Any = None, notes: str = "") -> Dict[str, Any]:
    return {
        "timestamp": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        "stage": stage,
        "seconds": seconds,
        "sites": SITES,
        "holes": HOLES,
        "nup": NUP,
        "ndown": NDOWN,
        "dim": PROFILE_DIM,
        "nnz": PROFILE_NNZ,
        "J_over_t": J_OVER_T,
        "n_neurons": N_NEURONS,
        "optimizer": OPTIMIZER_NAME,
        "loss": "" if loss is None else loss,
        "cuda_device": torch.cuda.get_device_name(TORCH_DEVICE),
        "H_type": PROFILE_H_TYPE,
        "input_type": PROFILE_INPUT_TYPE,
        "notes": notes,
    }

## Build model, Hamiltonian, inputs, and network

These setup timings are not per-epoch costs, but they separate CPU setup from the GPU training epoch. By default the notebook avoids `model.reorder_basis()` and skips unused next-nearest-neighbor Hamiltonian pieces.

In [20]:
def initialize_hashmap(model):
    model.hashmap = {tuple(state): idx for idx, state in enumerate(model.basis)}
    model.hashmap[tuple(np.zeros(model.M, dtype=np.byte))] = -1
    return model.hashmap


def hamiltonian_nn_only(model, basis, offset=0, verbose=False):
    Ht_row, Ht_col, Ht_val = [], [], []
    HJ_row, HJ_col, HJ_val = [], [], []
    neighbours = model.kneighbours(1)

    for local_col, state in enumerate(basis):
        col = local_col + offset
        diag = 0.0
        for i, j in neighbours:
            for sigma in (1, -1):
                newstate, sign = model.cicjdagger(sigma, i, j, state)
                row = model.hashmap.get(newstate, -1)
                if row != -1:
                    Ht_row.append(row)
                    Ht_col.append(col)
                    Ht_val.append(sign)

            newstate = model.XiXj(i, j, state)
            row = model.hashmap.get(newstate, -1)
            if row != -1:
                HJ_row.append(row)
                HJ_col.append(col)
                HJ_val.append(-0.5)

            diag += model.ZiZj(i, j, state) - 0.25 * model.ninj(i, j, state)

        if diag != 0:
            HJ_row.append(col)
            HJ_col.append(col)
            HJ_val.append(diag)

        if verbose and (col + 1) % 5000 == 0:
            print(f"Saved matrix columns {col + 1}/{model.dim}", flush=True)

    Ht = sparse.coo_matrix((Ht_val, (Ht_row, Ht_col)), shape=(model.dim, model.dim)).tocsr()
    HJ = sparse.coo_matrix((HJ_val, (HJ_row, HJ_col)), shape=(model.dim, model.dim)).tocsr()
    Ht = (Ht + Ht.T).tocsr()
    return Ht, HJ


TJNetRC.seed_torch(seed=RNG_SEED)
setup_rows = []

t0 = time.perf_counter()
model = tJ(NUP, NDOWN, LATTICE_SPEC["m"], LATTICE_SPEC["n"], LATTICE_SPEC["m_"], LATTICE_SPEC["n_"])
model_seconds = time.perf_counter() - t0
assert model.M == SITES
assert model.h == HOLES
assert len(model.kneighbours(1)) == 3 * model.M // 2

t0 = time.perf_counter()
if USE_REORDERED_BASIS:
    model.reorder_basis()
else:
    initialize_hashmap(model)
hashmap_seconds = time.perf_counter() - t0

print(f"Assembling nearest-neighbor tJ Hamiltonian: dim={model.dim:,}, sites={model.M}, holes={model.h}", flush=True)
t0 = time.perf_counter()
if USE_NN_ONLY_HAMILTONIAN:
    Ht, HJ = hamiltonian_nn_only(model, model.basis, verbose=False)
else:
    hamiltonian_terms = model.Hamiltonian(model.basis, verbose=False)
    Ht, _Ht2, HJ, _HJ2 = model.assembleH(hamiltonian_terms)
H_scipy = (TCOEF * Ht + JCOEF * HJ).astype(DTYPE_NP).tocsr()
assembly_seconds = time.perf_counter() - t0
PROFILE_DIM = model.dim
PROFILE_NNZ = H_scipy.nnz
H_scipy_type = type(H_scipy).__name__
Ht = HJ = None
if not USE_NN_ONLY_HAMILTONIAN:
    hamiltonian_terms = None
    _Ht2 = _HJ2 = None
release_memory()

input_data = None
_, input_seconds = timed_cuda(lambda: globals().__setitem__("input_data", TJNetRC.NN_inputs(TCOEF, JCOEF, model, trunc=None)))
PROFILE_INPUT_TYPE = str(input_data.dtype)
release_memory()

H_torch = None
_, H_transfer_seconds = timed_cuda(lambda: globals().__setitem__("H_torch", TJNetRC.processH(H_scipy, TORCH_DEVICE, PARAMS)))
PROFILE_H_TYPE = str(H_torch.layout)
if DROP_SCIPY_HAMILTONIAN_AFTER_GPU_TRANSFER:
    H_scipy = None
release_memory()

neuralnet = None
optimizer = None

def setup_network_optimizer():
    global neuralnet, optimizer
    net = TJNetRC.NeuralNetwork(model.M, N_NEURONS, dropout=DROPOUT)
    if USE_DATA_PARALLEL:
        net = nn.DataParallel(net)
    net.to(TORCH_DEVICE)
    neuralnet = net
    optimizer = TJNetRC.build_optimizer(PARAMS, neuralnet.parameters())
    return neuralnet

_, network_seconds = timed_cuda(setup_network_optimizer)

setup_rows.append(base_profile_row("model_basis", model_seconds, notes="CPU model and basis construction"))
setup_rows.append(base_profile_row("hashmap_or_reorder", hashmap_seconds, notes=f"USE_REORDERED_BASIS={USE_REORDERED_BASIS}"))
setup_rows.append(base_profile_row("hamiltonian_assembly_cpu", assembly_seconds, notes=f"USE_NN_ONLY_HAMILTONIAN={USE_NN_ONLY_HAMILTONIAN}"))
setup_rows.append(base_profile_row("input_to_gpu", input_seconds, notes="tJNetRC.NN_inputs including GPU transfer"))
setup_rows.append(base_profile_row("hamiltonian_to_gpu", H_transfer_seconds, notes="tJNetRC.processH sparse COO conversion and GPU transfer"))
setup_rows.append(base_profile_row("network_to_gpu_optimizer_setup", network_seconds, notes=f"NeuralNetwork, DataParallel={USE_DATA_PARALLEL}, to(cuda), build_optimizer"))

setup_summary = {
    "dim": PROFILE_DIM,
    "nnz": PROFILE_NNZ,
    "H_scipy_type": H_scipy_type,
    "H_scipy_retained": H_scipy is not None,
    "H_torch_layout": PROFILE_H_TYPE,
    "input_shape": tuple(input_data.shape),
    "input_dtype": PROFILE_INPUT_TYPE,
    "network_type": type(neuralnet).__name__,
    "setup_rows": setup_rows,
}
setup_summary

Assembling nearest-neighbor tJ Hamiltonian: dim=411,840, sites=16, holes=2


{'dim': 411840,
 'nnz': 6754176,
 'H_scipy_type': 'csr_matrix',
 'H_scipy_retained': False,
 'H_torch_layout': 'torch.sparse_coo',
 'input_shape': (411840, 18),
 'input_dtype': 'torch.float32',
 'network_type': 'DataParallel',
 'setup_rows': [{'timestamp': '2026-05-15T11:20:24',
   'stage': 'model_basis',
   'seconds': 0.6092361751943827,
   'sites': 16,
   'holes': 2,
   'nup': 7,
   'ndown': 7,
   'dim': 411840,
   'nnz': 6754176,
   'J_over_t': 0.3,
   'n_neurons': 128,
   'optimizer': 'Adam',
   'loss': '',
   'cuda_device': 'NVIDIA A100-SXM4-40GB MIG 3g.20gb',
   'H_type': 'torch.sparse_coo',
   'input_type': 'torch.float32',
   'notes': 'CPU model and basis construction'},
  {'timestamp': '2026-05-15T11:20:24',
   'stage': 'hashmap_or_reorder',
   'seconds': 0.6176491971127689,
   'sites': 16,
   'holes': 2,
   'nup': 7,
   'ndown': 7,
   'dim': 411840,
   'nnz': 6754176,
   'J_over_t': 0.3,
   'n_neurons': 128,
   'optimizer': 'Adam',
   'loss': '',
   'cuda_device': 'NVIDIA A10

In [23]:
def profile_forward(net):
    return net(input_data.float())


def profile_loss(net):
    psi = profile_forward(net)
    return TJNetRC.variational(H_torch, psi)


def gradient_step_only(net, optimizer):
    optimizer.zero_grad(set_to_none=True)
    loss = profile_loss(net)
    loss.backward(retain_graph=PROFILE_RETAIN_GRAPH)
    detached = loss.detach()
    del loss
    return detached


def update_only(optimizer):
    optimizer.step()
    return None


def actual_training_epoch_with_closure(net, optimizer):
    last_loss = {"value": None}

    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = profile_loss(net)
        loss.backward(retain_graph=PROFILE_RETAIN_GRAPH)
        last_loss["value"] = loss.detach()
        return loss

    optimizer.step(closure)
    return last_loss["value"]

## Optional warmup epoch

The warmup is not included in the profile rows. It initializes CUDA kernels and PyTorch autograd paths before the measured epoch.

In [24]:
warmup_seconds = None
warmup_loss = None

if RUN_WARMUP_EPOCH:
    warmup_loss, warmup_seconds = timed_cuda(lambda: actual_training_epoch_with_closure(neuralnet, optimizer))
    warmup_loss = scalar_float(warmup_loss)
release_memory()

{
    "RUN_WARMUP_EPOCH": RUN_WARMUP_EPOCH,
    "warmup_seconds": warmup_seconds,
    "warmup_loss": warmup_loss,
}

{'RUN_WARMUP_EPOCH': True,
 'warmup_seconds': 0.37618697714060545,
 'warmup_loss': -5.230141639709473}

## Profile one epoch

`forward_only` and `variational_given_psi` are diagnostic subpieces. `gradient_forward_loss_backward + optimizer_update` decomposes an Adam-style epoch. `actual_optimizer_step_closure` mirrors the exact `trainNN_transfer` call shape.

In [25]:
profile_rows = list(setup_rows)
release_memory()

if PROFILE_DIAGNOSTIC_SUBPIECES:
    grad_context = torch.enable_grad if PROFILE_DIAGNOSTIC_TRACK_GRAD else torch.no_grad
    diagnostic_note = "with autograd tracking" if PROFILE_DIAGNOSTIC_TRACK_GRAD else "under torch.no_grad()"

    with grad_context():
        psi_forward, forward_seconds = timed_cuda(lambda: profile_forward(neuralnet))
    profile_rows.append(base_profile_row("forward_only", forward_seconds, notes=f"neuralnet(input_data.float()) {diagnostic_note}"))

    with grad_context():
        loss_given_psi, variational_seconds = timed_cuda(lambda: TJNetRC.variational(H_torch, psi_forward))
    profile_rows.append(base_profile_row(
        "variational_given_psi",
        variational_seconds,
        loss=scalar_float(loss_given_psi),
        notes="sparse H@psi plus Rayleigh quotient, no NN forward/backward",
    ))
    del psi_forward, loss_given_psi
    release_memory()
else:
    profile_rows.append(base_profile_row(
        "diagnostic_subpieces_skipped",
        0.0,
        notes="Set PROFILE_DIAGNOSTIC_SUBPIECES=True to time forward_only and variational_given_psi.",
    ))

loss_backward, gradient_seconds = timed_cuda(lambda: gradient_step_only(neuralnet, optimizer))
profiled_loss = scalar_float(loss_backward)
del loss_backward
release_memory()
profile_rows.append(base_profile_row(
    "gradient_forward_loss_backward",
    gradient_seconds,
    loss=profiled_loss,
    notes=f"forward + variational + loss.backward(retain_graph={PROFILE_RETAIN_GRAPH})",
))

_, update_seconds = timed_cuda(lambda: update_only(optimizer))
release_memory()
profile_rows.append(base_profile_row("optimizer_update", update_seconds, loss=profiled_loss, notes="optimizer.step() after precomputed grads"))

profile_rows.append(base_profile_row(
    "profiled_epoch_total",
    gradient_seconds + update_seconds,
    loss=profiled_loss,
    notes="gradient_forward_loss_backward + optimizer_update",
))

if PROFILE_ACTUAL_CLOSURE_STEP:
    closure_loss, closure_seconds = timed_cuda(lambda: actual_training_epoch_with_closure(neuralnet, optimizer))
    closure_loss_scalar = scalar_float(closure_loss)
    del closure_loss
    release_memory()
    profile_rows.append(base_profile_row(
        "actual_optimizer_step_closure",
        closure_seconds,
        loss=closure_loss_scalar,
        notes="Exact trainNN_transfer shape: optimizer.step(closure)",
    ))
else:
    profile_rows.append(base_profile_row(
        "actual_optimizer_step_closure_skipped",
        0.0,
        notes="Set PROFILE_ACTUAL_CLOSURE_STEP=True to time an additional full closure epoch.",
    ))

profile_rows

[{'timestamp': '2026-05-15T11:20:24',
  'stage': 'model_basis',
  'seconds': 0.6092361751943827,
  'sites': 16,
  'holes': 2,
  'nup': 7,
  'ndown': 7,
  'dim': 411840,
  'nnz': 6754176,
  'J_over_t': 0.3,
  'n_neurons': 128,
  'optimizer': 'Adam',
  'loss': '',
  'cuda_device': 'NVIDIA A100-SXM4-40GB MIG 3g.20gb',
  'H_type': 'torch.sparse_coo',
  'input_type': 'torch.float32',
  'notes': 'CPU model and basis construction'},
 {'timestamp': '2026-05-15T11:20:24',
  'stage': 'hashmap_or_reorder',
  'seconds': 0.6176491971127689,
  'sites': 16,
  'holes': 2,
  'nup': 7,
  'ndown': 7,
  'dim': 411840,
  'nnz': 6754176,
  'J_over_t': 0.3,
  'n_neurons': 128,
  'optimizer': 'Adam',
  'loss': '',
  'cuda_device': 'NVIDIA A100-SXM4-40GB MIG 3g.20gb',
  'H_type': 'torch.sparse_coo',
  'input_type': 'torch.float32',
  'notes': 'USE_REORDERED_BASIS=False'},
 {'timestamp': '2026-05-15T11:20:24',
  'stage': 'hamiltonian_assembly_cpu',
  'seconds': 122.25668169208802,
  'sites': 16,
  'holes': 2,
 

In [26]:
append_profile_rows(RESULTS_PATH, profile_rows)

{
    "RESULTS_PATH": str(RESULTS_PATH),
    "rows": profile_rows,
}

{'RESULTS_PATH': '/n/holylabs/kaxiras_lab/axzhu/projects/tJ/profile_results/gpu_one_epoch_profile_python.tsv',
 'rows': [{'timestamp': '2026-05-15T11:20:24',
   'stage': 'model_basis',
   'seconds': 0.6092361751943827,
   'sites': 16,
   'holes': 2,
   'nup': 7,
   'ndown': 7,
   'dim': 411840,
   'nnz': 6754176,
   'J_over_t': 0.3,
   'n_neurons': 128,
   'optimizer': 'Adam',
   'loss': '',
   'cuda_device': 'NVIDIA A100-SXM4-40GB MIG 3g.20gb',
   'H_type': 'torch.sparse_coo',
   'input_type': 'torch.float32',
   'notes': 'CPU model and basis construction'},
  {'timestamp': '2026-05-15T11:20:24',
   'stage': 'hashmap_or_reorder',
   'seconds': 0.6176491971127689,
   'sites': 16,
   'holes': 2,
   'nup': 7,
   'ndown': 7,
   'dim': 411840,
   'nnz': 6754176,
   'J_over_t': 0.3,
   'n_neurons': 128,
   'optimizer': 'Adam',
   'loss': '',
   'cuda_device': 'NVIDIA A100-SXM4-40GB MIG 3g.20gb',
   'H_type': 'torch.sparse_coo',
   'input_type': 'torch.float32',
   'notes': 'USE_REORDERED_BA

## Optional PyTorch profiler hook

The cell below is intentionally commented out. Uncomment it if you want a CUDA-level PyTorch profile of one exact training-style epoch.

In [ ]:
# with torch.profiler.profile(
#     activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
#     record_shapes=True,
#     profile_memory=True,
# ) as prof:
#     actual_training_epoch_with_closure(neuralnet, optimizer)
#     torch.cuda.synchronize()
# print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=25))